# Module 3: Deploy

> Part of the **Modular Workshops** series. Standalone, ~25 min.

Moving the agent to production requires two things:

- **A governance layer** — workspace-level policies that govern every model call (PII and secrets redaction, allow-lists, cost caps). Currently facilitated by True Foundry.
- **Production-ready deployment infrastructure** — managed runtime for the agent. **LangSmith Deployments** gives 30+ endpoints, persistence, HITL, and Studio out of the box.

This module ships the policy-aware agent.

## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("LANGSMITH_API_KEY_GATEWAY set:", bool(os.environ.get("LANGSMITH_API_KEY_GATEWAY")))
print("WORKSPACE_ID set:             ", bool(os.environ.get("WORKSPACE_ID")))
print("OPENAI_API_KEY set:           ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

LANGSMITH_API_KEY set:         True
LANGSMITH_API_KEY_GATEWAY set: True
WORKSPACE_ID set:              True
OPENAI_API_KEY set:            True
TAVILY_API_KEY set:            True


---
# Part 1. Deploy — Ship the (Now-Governed) Agent

Policies are in place workspace-wide and the module's default model client routes through the gateway. Next, ship the deployment itself.

We deploy the agent in `agents/deep_agent/` to **LangSmith Deployments** with the `langgraph` CLI. Because `agents/deep_agent/agent.py` imports `model` from `utils.models`, the deployed image picks up the gateway `base_url` automatically — no extra flags, no separate config.

## 1.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`.


In [2]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


langgraph.json (workshop root)
---
deep_agent/
  deepagents.toml
  agent.py
  AGENTS.md
  skills/
    linkedin-post/
      SKILL.md
    twitter-post/
      SKILL.md


In [3]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


langgraph.json:
{
  "dependencies": ["."],
  "graphs": {
    "deep_agent": "./agents/deep_agent/agent.py:agent"
  },
  "env": ".env"
}

AGENTS.md:
# Research Assistant

You are an expert research assistant that can search the web, synthesize findings, and produce polished reports and content.

## Workflow

1. **Plan** — Use `write_todos` to break the task into steps
2. **Research** — Delegate research to the `research-agent` using the `task()` tool
3. **Synthesize** — Combine findings into a comprehensive report
4. **Write** — Save the final report to `/final_report.md`
5. **Remember** — Save key takeaways to `/memories/research_notes.md` for future reference

## Rules

- Delegate research to the research-agent rather than searching directly
- After receiving research results, synthesize and write the report yourself
- Consolidate citations — each unique URL gets one number [1], [2], [3]
- End reports with a Sources section listing all referenced URLs
- Check for relevant skills when a

The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 1.3 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 1.4 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


## 1.5 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> The image we're shipping picks up the gateway `base_url` from `utils/models.py` — no extra deploy flags needed. The deployed agent is governed by the policy from §1.1 from the moment it serves its first request.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name modular-workshops-deep-agent --no-input


## 1.6 What you get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


## Recap

| Pillar | What | How |
|---|---|---|
| **Govern** | Workspace-level PII / secrets redaction | `POST /v1/platform/gateway-policies` (`action=block`) |
| **Govern** | Route model calls through the gateway | `init_chat_model(..., base_url="<True Foundry Gateway URL>")` in `utils/models.py` |
| **Deploy** | Deployable graph config | `langgraph.json` at repo root |
| **Deploy** | Agent identity + skills | `AGENTS.md`, `skills/` |
| **Deploy** | Ship it | `langgraph deploy --name <your-deployment-name>` |

**The production pattern:** one policy + one `base_url` change + one deploy command. The agent now lives behind a managed server, every model call passes through a governed gateway, and 30+ endpoints are available to call it.

**Next:** Module 4 — LangSmith (tracing, querying traces, offline + online evals, annotation queues).